# Capítulo 7: Agentes de IA

## Objetivos de aprendizaje

- Comprender qué son los agentes de IA y cómo difieren de los LLMs simples.
- Conocer los componentes principales de un agente: razonamiento, herramientas y memoria.
- Implementar un agente básico con acceso a herramientas externas.
- Explorar patrones de diseño de agentes y sus aplicaciones.

## 7.1 ¿Qué es un Agente de IA?

Un **Agente de IA** es un sistema que utiliza un LLM como motor de razonamiento para tomar decisiones, ejecutar acciones y alcanzar objetivos de forma autónoma.

### LLM simple vs. Agente

| Aspecto | LLM simple | Agente |
|---------|-----------|--------|
| **Entrada/Salida** | Texto → Texto | Objetivo → Acciones → Resultado |
| **Herramientas** | Ninguna | Búsqueda, código, APIs, bases de datos |
| **Memoria** | Solo el contexto actual | Corto y largo plazo |
| **Planificación** | No planifica | Descompone tareas y planifica pasos |
| **Iteración** | Respuesta única | Ciclos de razonamiento-acción |

### Componentes de un agente

```
                    ┌─────────────┐
                    │   Objetivo   │
                    └──────┬──────┘
                           ▼
    ┌──────────────────────────────────────┐
    │         Motor de Razonamiento         │
    │              (LLM)                    │
    │                                      │
    │  Observar → Pensar → Actuar → Repetir│
    └──────┬───────────────┬───────────────┘
           ▼               ▼
    ┌────────────┐  ┌─────────────┐
    │Herramientas│  │   Memoria   │
    │ - Búsqueda │  │ - Corto plazo│
    │ - Código   │  │ - Largo plazo│
    │ - APIs     │  │ - Vectorial  │
    └────────────┘  └─────────────┘
```

## 7.2 El patrón ReAct: Reasoning + Acting

**ReAct** (Yao et al., 2023) es un patrón donde el agente alterna entre razonamiento y acción:

1. **Thought** (Pensamiento): El agente razona sobre la situación actual.
2. **Action** (Acción): Decide qué herramienta usar y con qué parámetros.
3. **Observation** (Observación): Recibe el resultado de la acción.
4. **Repeat**: Repite hasta completar la tarea.

### Ejemplo de traza ReAct

```
Pregunta: ¿Cuál es la capital del país más grande de Sudamérica?

Thought: Necesito encontrar el país más grande de Sudamérica.
Action: buscar("país más grande de Sudamérica")
Observation: Brasil es el país más grande de Sudamérica.

Thought: Ahora necesito encontrar la capital de Brasil.
Action: buscar("capital de Brasil")
Observation: La capital de Brasil es Brasilia.

Thought: Ya tengo la respuesta.
Respuesta: Brasilia es la capital de Brasil, el país más grande de Sudamérica.
```

In [ ]:
import json
from datetime import datetime

# Simulación de un agente con herramientas

# Definir herramientas disponibles
def herramienta_calculadora(expresion: str) -> str:
    """Evalúa una expresión matemática."""
    try:
        resultado = eval(expresion)
        return f"Resultado: {resultado}"
    except Exception as e:
        return f"Error: {e}"

def herramienta_fecha() -> str:
    """Retorna la fecha y hora actual."""
    return f"Fecha actual: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

def herramienta_buscar_base_datos(consulta: str) -> str:
    """Simula una búsqueda en una base de datos de productos."""
    productos = {
        "laptop": {"precio": 899990, "stock": 15, "marca": "TechPro"},
        "teclado": {"precio": 45990, "stock": 50, "marca": "KeyMax"},
        "mouse": {"precio": 25990, "stock": 80, "marca": "ClickPro"},
        "monitor": {"precio": 349990, "stock": 8, "marca": "ViewMax"}
    }
    consulta_lower = consulta.lower()
    for nombre, info in productos.items():
        if nombre in consulta_lower:
            return json.dumps({nombre: info}, ensure_ascii=False)
    return "Producto no encontrado."

HERRAMIENTAS = {
    "calculadora": herramienta_calculadora,
    "fecha": herramienta_fecha,
    "buscar_producto": herramienta_buscar_base_datos
}

print("Herramientas disponibles:")
for nombre, func in HERRAMIENTAS.items():
    print(f"  - {nombre}: {func.__doc__}")

In [ ]:
# Implementación simplificada de un agente ReAct

class AgenteSimple:
    """Agente simplificado que demuestra el patrón ReAct."""
    
    def __init__(self, herramientas):
        self.herramientas = herramientas
        self.historial = []
    
    def pensar(self, objetivo, observacion=None):
        """Simula el paso de razonamiento del agente."""
        paso = {"tipo": "pensamiento"}
        if observacion:
            paso["contexto"] = observacion
        paso["objetivo"] = objetivo
        self.historial.append(paso)
        return paso
    
    def actuar(self, herramienta, parametro=None):
        """Ejecuta una herramienta y registra la observación."""
        if herramienta not in self.herramientas:
            resultado = f"Herramienta '{herramienta}' no disponible."
        elif parametro:
            resultado = self.herramientas[herramienta](parametro)
        else:
            resultado = self.herramientas[herramienta]()
        
        paso = {
            "tipo": "acción",
            "herramienta": herramienta,
            "parametro": parametro,
            "resultado": resultado
        }
        self.historial.append(paso)
        return resultado
    
    def mostrar_historial(self):
        """Muestra la traza de ejecución del agente."""
        for i, paso in enumerate(self.historial, 1):
            if paso["tipo"] == "pensamiento":
                print(f"  Paso {i} [Pensamiento]: {paso['objetivo']}")
            elif paso["tipo"] == "acción":
                print(f"  Paso {i} [Acción]: {paso['herramienta']}({paso['parametro']})")
                print(f"          [Resultado]: {paso['resultado']}")

# Demostración del agente
agente = AgenteSimple(HERRAMIENTAS)

print("Objetivo: ¿Cuál es el precio total de 3 laptops con IVA (19%)?\n")

# Paso 1: Buscar el precio de la laptop
agente.pensar("Primero necesito buscar el precio de una laptop")
resultado = agente.actuar("buscar_producto", "laptop")

# Paso 2: Calcular el total con IVA
agente.pensar("El precio es $899.990. Ahora calculo 3 unidades con IVA del 19%")
resultado = agente.actuar("calculadora", "899990 * 3 * 1.19")

# Paso 3: Obtener la fecha
agente.pensar("Registro la fecha de la cotización")
resultado = agente.actuar("fecha")

print("Traza de ejecución:")
agente.mostrar_historial()

## 7.3 Memoria del agente

La **memoria** permite al agente retener información entre interacciones.

### Tipos de memoria

| Tipo | Descripción | Implementación |
|------|-------------|----------------|
| **Corto plazo** | Historial de la conversación actual | Lista de mensajes |
| **Largo plazo** | Información persistente entre sesiones | Base de datos vectorial |
| **Episódica** | Experiencias pasadas específicas | Retrievals de ejemplos similares |
| **Semántica** | Conocimiento general extraído | Grafos de conocimiento |

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

class MemoriaVectorial:
    """Memoria basada en búsqueda por similitud semántica."""
    
    def __init__(self):
        self.documentos = []
        self.metadatos = []
        self.vectorizer = TfidfVectorizer()
        self._vectores = None
    
    def agregar(self, texto, metadata=None):
        """Agrega un documento a la memoria."""
        self.documentos.append(texto)
        self.metadatos.append(metadata or {})
        self._vectores = None  # Invalidar cache
    
    def buscar(self, consulta, top_k=3):
        """Busca los documentos más relevantes."""
        if not self.documentos:
            return []
        if self._vectores is None:
            self._vectores = self.vectorizer.fit_transform(self.documentos)
        
        query_vec = self.vectorizer.transform([consulta])
        similitudes = cosine_similarity(query_vec, self._vectores).flatten()
        indices = similitudes.argsort()[::-1][:top_k]
        
        resultados = []
        for idx in indices:
            if similitudes[idx] > 0:
                resultados.append({
                    "texto": self.documentos[idx],
                    "similitud": similitudes[idx],
                    "metadata": self.metadatos[idx]
                })
        return resultados

# Ejemplo de uso
memoria = MemoriaVectorial()

# Agregar conocimiento a la memoria
memoria.agregar("Python es un lenguaje de programación de alto nivel", {"tema": "programación"})
memoria.agregar("TF-IDF pondera la importancia de términos en documentos", {"tema": "NLP"})
memoria.agregar("K-Means agrupa datos en clusters basados en similitud", {"tema": "ML"})
memoria.agregar("BERT es un modelo Transformer pre-entrenado bidireccionalmente", {"tema": "NLP"})
memoria.agregar("Los agentes de IA combinan LLMs con herramientas externas", {"tema": "agentes"})

# Buscar en la memoria
resultados = memoria.buscar("modelo de lenguaje para procesamiento de texto")
print("Búsqueda: 'modelo de lenguaje para procesamiento de texto'\n")
for r in resultados:
    print(f"  [{r['metadata'].get('tema', 'N/A')}] (sim={r['similitud']:.3f}) {r['texto']}")

## 7.4 RAG: Retrieval Augmented Generation

**RAG** combina la recuperación de información relevante con la generación de texto. Es una de las arquitecturas más utilizadas para dar a los LLMs acceso a conocimiento actualizado o específico.

### Pipeline RAG

```
Pregunta del usuario
    │
    ▼
Buscar documentos relevantes en la base de conocimiento
    │
    ▼
Construir prompt: contexto recuperado + pregunta
    │
    ▼
LLM genera respuesta basada en el contexto
    │
    ▼
Respuesta fundamentada en fuentes
```

In [ ]:
# Simulación de un pipeline RAG simplificado

class RAGSimple:
    """Pipeline RAG simplificado."""
    
    def __init__(self, base_conocimiento):
        self.memoria = MemoriaVectorial()
        for doc in base_conocimiento:
            self.memoria.agregar(doc)
    
    def generar_prompt(self, pregunta, contextos):
        """Construye el prompt con contexto recuperado."""
        contexto_str = "\n".join([f"- {c['texto']}" for c in contextos])
        prompt = f"""Basándote en la siguiente información:

{contexto_str}

Responde la pregunta: {pregunta}"""
        return prompt
    
    def responder(self, pregunta):
        """Pipeline completo: recuperar + generar prompt."""
        # 1. Recuperar contexto relevante
        contextos = self.memoria.buscar(pregunta, top_k=2)
        
        # 2. Generar prompt
        prompt = self.generar_prompt(pregunta, contextos)
        
        return {
            "pregunta": pregunta,
            "contextos": contextos,
            "prompt_generado": prompt
        }

# Base de conocimiento
base = [
    "La empresa TechCorp fue fundada en 2015 en Santiago de Chile.",
    "TechCorp ofrece soluciones de inteligencia artificial para empresas.",
    "El producto principal de TechCorp es un chatbot llamado AsistenteIA.",
    "AsistenteIA puede manejar hasta 10.000 consultas simultáneas.",
    "Los planes de TechCorp van desde $99.000 hasta $999.000 mensuales.",
    "TechCorp tiene oficinas en Chile, Colombia y México."
]

rag = RAGSimple(base)
resultado = rag.responder("¿Cuánto cuestan los servicios de TechCorp?")

print(f"Pregunta: {resultado['pregunta']}\n")
print("Contextos recuperados:")
for c in resultado['contextos']:
    print(f"  (sim={c['similitud']:.3f}) {c['texto']}")
print(f"\nPrompt generado:\n{resultado['prompt_generado']}")

## 7.5 Patrones de diseño de agentes

| Patrón | Descripción | Ejemplo |
|--------|-------------|---------|
| **ReAct** | Razonamiento + Acción iterativa | Agente de búsqueda |
| **Plan-and-Execute** | Planifica todos los pasos antes de ejecutar | Agente de investigación |
| **Multi-agente** | Varios agentes especializados colaboran | Equipo de desarrollo |
| **Reflexión** | El agente revisa y corrige sus propias respuestas | Generación de código |
| **RAG** | Recuperación + Generación | Asistente con base de conocimiento |

## Resumen

En este capítulo aprendimos:

- **Agentes de IA**: Sistemas autónomos que combinan razonamiento, herramientas y memoria.
- **Patrón ReAct**: Ciclo de pensamiento → acción → observación.
- **Herramientas**: Los agentes pueden usar calculadoras, APIs, bases de datos y más.
- **Memoria**: Corto plazo (conversación) y largo plazo (base vectorial).
- **RAG**: Retrieval Augmented Generation para respuestas fundamentadas.

En el próximo capítulo exploraremos **casos de uso reales** de analítica textual en diversas industrias.